# Qwen3.5-0.8B FlyCore-v1 FROM trained FlyFFN-v3 standalone

This notebook starts from the already-trained Hugging Face model:

**vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN**

Instead of downloading base Qwen and retraining all 24 FlyFFNs, it:

1. downloads the verified FlyFFN-v3 standalone checkpoint from Hugging Face,
2. reconstructs the trained 24/24 FlyFFN-v3 model,
3. takes the FlyWire adjacency and routing state directly from that checkpoint,
4. factorizes only the existing embedding,
5. replaces only the embedding + LM head with FlyEmbedding + FlyLMHead,
6. calibrates the new vocabulary core,
7. performs a low-LR joint refinement,
8. runs the same FastEval and dual chat tests against the source FlyFFN-v3 model,
9. exports and uploads a new verified FlyCore standalone.

**No Zenodo graph download is used in this workflow.**


In [ ]:
#@title 1. Update repository, install dependencies, and preflight
import pathlib, subprocess, sys
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','datasets','huggingface_hub',
    'safetensors','ipywidgets','pandas','requests','tqdm'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0,str(SRC_DIR))

for p in [
    REPO_DIR/'scripts'/'run_qwen35_flycore_from_v3.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flycore_v1.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flycore_standalone.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_standalone.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

from tinycenn_lm.qwen35_standalone import load_standalone
from tinycenn_lm.qwen35_flycore_standalone import load_flycore_standalone
print('✓ FlyCore-from-v3 workflow imports OK')
print('Ready:',REPO_DIR)


In [ ]:
#@title 2. Configuration
SOURCE_REPO_ID = "vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN" #@param {type:'string'}
SOURCE_DIR = "/content/qwen35_v3_source" #@param {type:'string'}

RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}

VOCAB_LATENT_DIM = 384 #@param {type:'integer'}
VOCAB_GRAPH_MIX_INIT = 0.10 #@param {type:'number'}
FACTOR_CHUNK_ROWS = 4096 #@param {type:'integer'}

OUTPUT_DIR = REPO_DIR/'results'/'flycore_from_v3_qwen35_08b'

SAVE_STANDALONE = True #@param {type:'boolean'}
UPLOAD_TO_HF = True #@param {type:'boolean'}
HF_REPO_ID = "vtava/Qwen35-0.8B-FlyCore-v1-From-v3" #@param {type:'string'}
HF_PRIVATE = False #@param {type:'boolean'}

print('Source standalone:',SOURCE_REPO_ID)
print('No Zenodo download: True')
print('Preserve trained FlyFFN-v3 24/24: True')
print('Replace only embedding + LM head')
print('Vocabulary latent rank:',VOCAB_LATENT_DIM)
print('Output HF repo:',HF_REPO_ID)


In [ ]:
#@title 3. Download v3 standalone → train FlyEmbedding/FlyLMHead — live output
import os, subprocess, sys

cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flycore_from_v3.py'),
    '--source-repo-id',SOURCE_REPO_ID,
    '--source-dir',SOURCE_DIR,
    '--run-mode',RUN_MODE,
    '--seq-len',str(SEQ_LEN),
    '--batch-size',str(BATCH_SIZE),
    '--vocab-latent-dim',str(VOCAB_LATENT_DIM),
    '--vocab-graph-mix-init',str(VOCAB_GRAPH_MIX_INIT),
    '--factor-chunk-rows',str(FACTOR_CHUNK_ROWS),
    '--output-dir',str(OUTPUT_DIR),
]

if SAVE_STANDALONE:
    cmd += ['--standalone-dir',str(OUTPUT_DIR/'standalone')]

if UPLOAD_TO_HF:
    if not HF_REPO_ID.strip():
        raise ValueError('HF_REPO_ID is empty')
    from huggingface_hub import get_token, notebook_login
    token=os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        if not token:
            token=userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        token=get_token()
    if not token:
        notebook_login()
        token=get_token()
    if not token:
        raise RuntimeError('Hugging Face login failed')
    os.environ['HF_TOKEN']=token
    cmd += ['--upload-hf','--hf-repo-id',HF_REPO_ID]
    if HF_PRIVATE:
        cmd.append('--hf-private')

print('='*100)
print('FlyCore-v1 FROM trained FlyFFN-v3 standalone')
print('Source:',SOURCE_REPO_ID)
print('Command:',' '.join(cmd))
print('='*100)

env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
if os.environ.get('HF_TOKEN'):
    env['HF_TOKEN']=os.environ['HF_TOKEN']

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display

summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
display(summary)

print('\nSOURCE')
print('repo:',report['source_repo_id'])
print('trained v3 source:',report['source_is_trained_flyffn_v3'])
print('Zenodo graph required:',report['source_graph_download_required'])
print('Adjacency from checkpoint:',report['source_flywire_adjacency_from_checkpoint'])

print('\nVOCAB FACTORIZATION')
for k,v in report['factorization'].items():
    print(k,':',v)

print('\nINITIAL PROBE')
for k,v in report['initial_fly_vocab_probe'].items():
    print(k,':',v)

print('\nAFTER VOCAB CALIBRATION')
for k,v in report['vocab_calibration'].items():
    print(k,':',v)

print('\nVOCAB PARAMETER STATS')
for k,v in report['vocab_parameter_stats'].items():
    print(k,':',v)

print('\nKEY RESULTS VS SOURCE V3')
for k in [
    'ce_gap_vs_source_v3',
    'ppl_ratio_vs_source_v3',
    'parameter_ratio_flycore_over_source_v3',
    'decode_speed_ratio_flycore_over_source_v3',
]:
    print(k,':',report[k])

print('\nSOURCE ROUTING PRESERVED')
print('same layer set:',set(report['source_routing_schedule'])==set(report['final_routing_schedule']))


In [ ]:
#@title 5. Load source FlyFFN-v3 + trained FlyCore for FastEval and chat
import torch
from pathlib import Path
from huggingface_hub import snapshot_download
from tinycenn_lm.qwen35_standalone import load_standalone
from tinycenn_lm.qwen35_flycore_standalone import load_flycore_standalone

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=(
    torch.bfloat16
    if device.type=='cuda' and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type=='cuda' else torch.float32)
)

source_dir=Path(SOURCE_DIR)
if not (source_dir/'standalone_state.pt').exists():
    source_dir=Path(snapshot_download(
        repo_id=SOURCE_REPO_ID,
        repo_type='model',
        local_dir=SOURCE_DIR,
        allow_patterns=[
            'standalone_state.pt','standalone_manifest.json',
            'flyffn_config.json','config.json','generation_config.json',
            'tokenizer*','special_tokens_map.json','chat_template*','*.jinja'
        ],
        token=os.environ.get('HF_TOKEN') or None,
    ))

print('Loading source trained FlyFFN-v3...')
source_model,tokenizer=load_standalone(source_dir,device=device,dtype=dtype)

print('Loading trained FlyCore standalone...')
fly_model,fly_tokenizer=load_flycore_standalone(
    OUTPUT_DIR/'standalone',
    device=device,
    dtype=dtype,
)

# Keep the old variable name so the unchanged FastEval implementation can be reused.
qwen_model=source_model

print('✓ Source FlyFFN-v3 ready')
print('✓ FlyCore-v1 ready')
print('✓ Same tokenizer vocabulary:',tokenizer.vocab_size==fly_tokenizer.vocab_size)


In [ ]:
#@title 6. FastEval — 50 items: MMLU-Pro / PIQA / MMMLU-DE (GPQA optional)
import os, random, time
import pandas as pd, torch
from datasets import load_dataset
from IPython.display import display
N=50; EVAL_BATCH=4; MAX_LENGTH=1024; SEED=2026
LETTERS=list('ABCDEFGHIJ')

def sample(ds,n,seed): return ds.shuffle(seed=seed).select(range(min(n,len(ds))))
def prompt_mc(q,opts,german=False):
    labels=LETTERS[:len(opts)]
    lead=('Wähle die richtige Antwort. Antworte nur mit dem Buchstaben.' if german else 'Choose the correct answer. Reply only with the answer letter.')
    lines=[lead,'',('Frage: ' if german else 'Question: ')+str(q),'']+[f'{a}. {o}' for a,o in zip(labels,opts)]+['',('Antwort:' if german else 'Answer:')]
    return '\n'.join(lines),labels

def chat_wrap(text):
    return tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=False,add_generation_prompt=True)

def label_ids(labels):
    out=[]
    for a in labels:
        choices=[tokenizer.encode(' '+a,add_special_tokens=False),tokenizer.encode(a,add_special_tokens=False)]
        one=next((x[0] for x in choices if len(x)==1),None)
        if one is None: raise RuntimeError(f'Answer label {a} is not one token: {choices}')
        out.append(one)
    return out

benches={}
ds=sample(load_dataset('TIGER-Lab/MMLU-Pro',split='test'),N,SEED)
benches['MMLU-Pro']=[{'prompt':prompt_mc(x['question'],list(x['options']))[0],'labels':prompt_mc(x['question'],list(x['options']))[1],'gold':int(x['answer_index'])} for x in ds]
ds=sample(load_dataset('regisss/piqa',split='validation'),N,SEED+1)
benches['PIQA']=[{'prompt':prompt_mc(x['goal'],[x['sol1'],x['sol2']])[0],'labels':['A','B'],'gold':int(x['label'])} for x in ds]
try: ds=load_dataset('openai/MMMLU','DE_DE',split='test')
except Exception: ds=load_dataset('openai/MMMLU',split='test')
ds=sample(ds,N,SEED+2); mm=[]
for x in ds:
    opts=[str(x[k]) for k in ['A','B','C','D']]; p,l=prompt_mc(str(x['Question']),opts,True); mm.append({'prompt':p,'labels':l,'gold':l.index(str(x['Answer']).strip().upper())})
benches['MMMLU-DE']=mm
HF_TOKEN=os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    if not HF_TOKEN: HF_TOKEN=userdata.get('HF_TOKEN')
except: pass
try:
    ds=sample(load_dataset('Idavidrein/gpqa','gpqa_diamond',split='train',token=HF_TOKEN),N,SEED+3); gp=[]
    for i,x in enumerate(ds):
        raw=[x['Correct Answer'],x['Incorrect Answer 1'],x['Incorrect Answer 2'],x['Incorrect Answer 3']]; order=list(range(4)); random.Random(SEED+10000+i).shuffle(order)
        opts=[raw[j] for j in order]; p,l=prompt_mc(x['Question'],opts); gp.append({'prompt':p,'labels':l,'gold':order.index(0)})
    benches['GPQA-Diamond']=gp
except Exception as e: print('GPQA skipped:',str(e)[:140])

@torch.inference_mode()
def run_eval(model,name):
    ans={}; model.eval(); tokenizer.padding_side='right'; tokenizer.truncation_side='left'
    print('\n'+'='*92+'\nMODEL:',name+'\n'+'='*92)
    for bn,items in benches.items():
        correct=done=0; t0=time.perf_counter()
        for s in range(0,len(items),EVAL_BATCH):
            batch=items[s:s+EVAL_BATCH]; texts=[chat_wrap(x['prompt']) for x in batch]
            enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=MAX_LENGTH).to(device)
            last=enc.attention_mask.sum(1)-1
            out=model(**enc,use_cache=False,return_dict=True).logits.float()
            for j,x in enumerate(batch):
                ids=torch.tensor(label_ids(x['labels']),device=device); pred=int(out[j,int(last[j])][ids].argmax())
                correct+=pred==x['gold']; done+=1
            if done%10==0 or done==len(items): print(f'  {done:>2}/{len(items)} | correct={correct:>2} | acc={100*correct/done:5.1f}% | {done/max(time.perf_counter()-t0,1e-9):5.2f} q/s')
        ans[bn]={'correct':correct,'total':done,'accuracy':correct/done}; print(f'  DONE → {correct}/{done} = {100*correct/done:.1f}%')
    return ans

base_eval=run_eval(qwen_model,'Source FlyFFN-v3')
fly_eval=run_eval(fly_model,'Source v3 FlyCore-v1')
rows=[]
for bn in benches:
    b,f=base_eval[bn],fly_eval[bn]; bp,fp=100*b['accuracy'],100*f['accuracy']
    rows.append({'Benchmark':bn,'v3 correct':f"{b['correct']}/{b['total']}",'v3 %':bp,'FlyCore-v1 correct':f"{f['correct']}/{f['total']}",'FlyCore-v1 %':fp,'Δ FlyCore-v1':fp-bp})
fast_eval_df=pd.DataFrame(rows).set_index('Benchmark')
display(fast_eval_df.style.format({'v3 %':'{:.1f}%','FlyCore-v1 %':'{:.1f}%','Δ FlyCore-v1':'{:+.1f}'}))
print(f"Macro: v3={fast_eval_df['v3 %'].mean():.1f}% | Fly={fast_eval_df['FlyCore-v1 %'].mean():.1f}% | Δ={fast_eval_df['Δ FlyCore-v1'].mean():+.1f} points")
fast_eval_df.to_csv(OUTPUT_DIR/'fast_eval_50_flycore_from_v3.csv')


In [ ]:
#@title 7. Interactive dual CHAT — source FlyFFN-v3 vs FlyCore-v1
import time, html, torch, ipywidgets as widgets
from IPython.display import display, HTML

base_history=[]; fly_history=[]

def _reply(model,history,user_text,max_new=192):
    msgs=history+[{'role':'user','content':user_text}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    enc=tokenizer(text,return_tensors='pt').to(device)
    t=time.perf_counter()
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id)
    dt=time.perf_counter()-t
    reply=tokenizer.decode(out[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
    return reply,dt

def ask_both(user_text):
    global base_history, fly_history
    b,bt=_reply(qwen_model,base_history,user_text); f,ft=_reply(fly_model,fly_history,user_text)
    base_history += [{'role':'user','content':user_text},{'role':'assistant','content':b}]
    fly_history  += [{'role':'user','content':user_text},{'role':'assistant','content':f}]
    return b,bt,f,ft

def reset_chat(*_):
    global base_history,fly_history
    base_history=[]; fly_history=[]; out.clear_output()
    with out: print('Chat reset ✓')

prompt=widgets.Textarea(value='Explain in simple terms how a sparse FFN can save computation, and give one possible risk.',description='You:',layout=widgets.Layout(width='100%',height='100px'))
ask=widgets.Button(description='Ask both models',button_style='success'); reset=widgets.Button(description='Reset chat')
out=widgets.Output()

def on_ask(_):
    q=prompt.value.strip()
    if not q: return
    with out:
        print('\n'+'='*100); print('USER:',q)
        b,bt,f,ft=ask_both(q)
        table=(f"<table style='width:100%;table-layout:fixed'><tr><th>Source FlyFFN-v3-0.8B ({bt:.2f}s)</th>"
               f"<th>FlyCore-v1 ({ft:.2f}s)</th></tr><tr><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>"
               f"{html.escape(b)}</td><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>{html.escape(f)}</td></tr></table>")
        display(HTML(table))
    prompt.value=''

ask.on_click(on_ask); reset.on_click(reset_chat)
display(widgets.VBox([prompt,widgets.HBox([ask,reset]),out]))
print('Dual multi-turn chat ready. Each model keeps its own conversation history.')


In [ ]:
#@title 8. Verify FlyCore standalone export
import json, torch

STANDALONE_DIR=OUTPUT_DIR/'standalone'
manifest=json.loads((STANDALONE_DIR/'standalone_manifest.json').read_text())
assert manifest.get('verified') is True

state=torch.load(
    STANDALONE_DIR/'standalone_state.pt',
    map_location='cpu',
    weights_only=True,
    mmap=True,
)
required=[
    'fly_vocab_core.codebook.weight',
    'fly_vocab_core.basis',
    'fly_vocab_core.fly_down.weight',
    'fly_vocab_core.fly_up.weight',
    'fly_vocab_core.adjacency',
    'flyffn_shared_graph.adjacency',
]
missing=[k for k in required if k not in state]
if missing:
    raise RuntimeError('Standalone missing FlyCore keys: '+str(missing))
del state

print('✓ FlyCore standalone verified')
print('State keys   :',manifest['state_keys'])
print('FlyCore keys :',manifest['flycore_state_keys'])
print('Size         :',f"{manifest['state_bytes']/1024**3:.3f} GB")
if UPLOAD_TO_HF:
    print('Hugging Face :',f'https://huggingface.co/{HF_REPO_ID}')


In [ ]:
#@title 9. Fresh FlyCore standalone reload + chat test
import gc, torch
from tinycenn_lm.qwen35_flycore_standalone import load_flycore_standalone

for _name in ['qwen_model','source_model','fly_model']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

standalone_model,standalone_tokenizer=load_flycore_standalone(
    OUTPUT_DIR/'standalone',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

messages=[{'role':'user','content':'Explain in simple terms what a neural network is.'}]
prompt=standalone_tokenizer.apply_chat_template(
    messages,tokenize=False,add_generation_prompt=True
)
inputs=standalone_tokenizer(prompt,return_tensors='pt').to(next(standalone_model.parameters()).device)

with torch.inference_mode():
    out=standalone_model.generate(
        **inputs,max_new_tokens=160,do_sample=False,use_cache=True,
        pad_token_id=standalone_tokenizer.eos_token_id,
    )

answer=standalone_tokenizer.decode(
    out[0,inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
).strip()

print('✓ Fresh standalone reconstruction succeeded')
print('\nFlyCore answer:\n',answer)


In [ ]:
#@title 10. Sync FastEval/results to Hugging Face
import os, shutil
from tinycenn_lm.qwen35_flycore_standalone import upload_flycore_standalone

STANDALONE_DIR=OUTPUT_DIR/'standalone'
eval_file=OUTPUT_DIR/'fast_eval_50_flycore_from_v3.csv'

if eval_file.exists():
    shutil.copy2(eval_file,STANDALONE_DIR/eval_file.name)
    print('✓ Added latest FastEval CSV')

if 'fast_eval_df' in globals():
    fast_eval_df.to_csv(STANDALONE_DIR/'fast_eval_50_flycore_from_v3.csv')

if UPLOAD_TO_HF:
    url=upload_flycore_standalone(
        STANDALONE_DIR,
        HF_REPO_ID,
        token=os.environ.get('HF_TOKEN'),
        private=HF_PRIVATE,
    )
    print('✓ Hugging Face synchronized:',url)
else:
    print('UPLOAD_TO_HF=False — standalone remains local at',STANDALONE_DIR)
